In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


df = pd.read_csv('df_processed.csv')
camry_data = pd.read_csv('camry_processed.csv')

X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

results = []

def evaluate_model(model_name, model, params_desc):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mse)
    
    results.append({
        'Модель': model_name,
        'Гиперпараметры': params_desc,
        'MAE': round(mae, 2),
        'MSE': round(mse, 2),
        'RMSE': round(rmse, 2),
        'R^2': round(r2, 4)
    })
    return model


                                                            #мин. примеров в листе      # доля признаков на каждое разбиение
rf_1 = RandomForestRegressor(n_estimators=100, max_depth=None, min_samples_leaf=1,      max_features=1.0, random_state=42, n_jobs=-1)
evaluate_model('Random Forest (1)', rf_1, 'n_est=100, depth=None, leaf=1, feat=1.0')
                                                            # мин. примеров для разбиения узла
rf_2 = RandomForestRegressor(n_estimators=100, max_depth=15, min_samples_split=5,               min_samples_leaf=2, max_features='sqrt', random_state=42, n_jobs=-1)
evaluate_model('Random Forest (2)', rf_2, 'n_est=100, depth=15, leaf=2, feat=sqrt')

rf_3 = RandomForestRegressor(n_estimators=300, max_depth=20, min_samples_split=5, min_samples_leaf=1, max_features=0.9, random_state=42, n_jobs=-1)
evaluate_model('Random Forest (3)', rf_3, 'n_est=300, depth=20, leaf=1, feat=0.9')

                                                                      # доля строк на каждое дерево # доля признаков на каждое дерево
xgb_1 = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, subsample=1.0,               colsample_bytree=1.0, random_state=42)
evaluate_model('XGBoost (1)', xgb_1, 'n_est=100, lr=0.1, depth=6, sub=1.0, col=1.0')

xgb_2 = XGBRegressor(n_estimators=150, learning_rate=0.1, max_depth=5, subsample=0.9, colsample_bytree=0.9, random_state=42)
evaluate_model('XGBoost (2)', xgb_2, 'n_est=150, lr=0.1, depth=5, sub=0.9, col=0.9')

xgb_3 = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=8, subsample=0.85, colsample_bytree=0.85, random_state=42)
evaluate_model('XGBoost (3)', xgb_3, 'n_est=300, lr=0.05, depth=8, sub=0.85, col=0.85')

results_df = pd.DataFrame(results)
print("\nТаблица результатов сравнения моделей:\n")
display(results_df) 

best_rf = rf_3 
best_xgb = xgb_3
print("\nПредсказание для отдельной машины (Camry):")
camry_features = camry_data.drop(columns=['price'])
real_price = camry_data['price'].iloc[0] 
camry_pred_rf = best_rf.predict(camry_features)[0]
camry_pred_xgb = best_xgb.predict(camry_features)[0]

print(f"Прогноз Random Forest: {camry_pred_rf:,.2f}")
print(f"Прогноз XGBoost:       {camry_pred_xgb:,.2f}")
print(f"Реальная цена:         {real_price:,.2f}")


Таблица результатов сравнения моделей:



,Модель,Гиперпараметры,MAE,MSE,RMSE,R^2
0,Random Forest (1),"n_est=100, depth=None, leaf=1, feat=1.0",119354.94,3.221659e+10,179489.80,0.9571
1,Random Forest (2),"n_est=100, depth=15, leaf=2, feat=sqrt",116613.73,2.946082e+10,171641.53,0.9608
2,Random Forest (3),"n_est=300, depth=20, leaf=1, feat=0.9",116577.47,3.011535e+10,173537.74,0.9599
3,XGBoost (1),"n_est=100, lr=0.1, depth=6, sub=1.0, col=1.0",115932.07,2.843099e+10,168614.92,0.9621
4,XGBoost (2),"n_est=150, lr=0.1, depth=5, sub=0.9, col=0.9",117298.89,2.877553e+10,169633.51,0.9617
5,XGBoost (3),"n_est=300, lr=0.05, depth=8, sub=0.85, col=0.85",112872.50,2.861994e+10,169174.27,0.9619



Предсказание для отдельной машины (Camry):
Прогноз Random Forest: 1,529,640.38
Прогноз XGBoost:       1,474,613.38
Реальная цена:         1,450,000.00


![Метрики качества (Эксперимент 2)](linear_metrics.png)

![Предсказание линейной регрессии для Camry](linear_camry.png)

In [2]:
np.sqrt(49097261315)

221579.01821923483